# 03 — Hallazgos

Esta notebook produce los **tres números** del pitch. Si no podés completar las
tres frases del final, el proyecto no está terminado.


In [ ]:
import sys; sys.path.insert(0, '..')
import json, pandas as pd
from src import config as cfg
from src import figuras
pd.set_option('display.float_format', lambda v: f'{v:,.1f}')

disp   = pd.read_csv(cfg.PROCESSED / 'fact_dispersion.csv', parse_dates=['fecha'])
canast = pd.read_csv(cfg.PROCESSED / 'fact_canasta.csv', parse_dates=['fecha'])
rank   = pd.read_csv(cfg.PROCESSED / 'fact_ranking_cadenas.csv', parse_dates=['fecha'])
h      = json.load(open(cfg.PROCESSED / 'resumen_hallazgos.json'))
h

## Hallazgo 1 — La misma góndola, precios muy distintos

Comparo el precio mediano de cada item **dentro de la misma provincia**, entre
cadenas. Comparar entre provincias mezclaría costos logísticos e impuestos.

In [ ]:
ult = disp[disp.fecha == disp.fecha.max()]
top = (ult.groupby('item')
          .agg(gap_pct=('gap_pct','median'), precio_min=('precio_min','median'),
               precio_max=('precio_max','median'))
          .sort_values('gap_pct', ascending=False))
display(top.head(15))
print(f"Gap mediano de toda la canasta: {ult.gap_pct.median():.1f}%")
figuras.fig_dispersion_items()

## Hallazgo 2 — Cuánto vale elegir bien

Traducido a plata: cuánto gasta por mes el hogar tipo según dónde compre.
El escenario "óptimo" no es alcanzable en la práctica (nadie recorre 6 cadenas),
pero define el techo del ahorro posible. Es la métrica que le importaría a una
app de comparación de precios.

In [ ]:
c = canast.groupby('fecha')[['canasta_optima','canasta_tipica','canasta_peor']].mean()
display(c.tail())
ult = c.iloc[-1]
print(f"Ahorro máximo: ${ult.canasta_tipica-ult.canasta_optima:,.0f}/mes "
      f"({100*(1-ult.canasta_optima/ult.canasta_tipica):.1f}%)")
figuras.fig_canasta_escenarios()

## Hallazgo 3 — Ranking de cadenas

Índice 100 = mediana del mercado en esa provincia y ese día. Uso mediana de
ratios y no ratio de medianas para que una cadena con surtido distinto no
distorsione el número.

In [ ]:
r = rank[rank.fecha == rank.fecha.max()]
display(r.groupby('cadena')['indice_vs_mercado'].agg(['median','min','max','count'])
         .sort_values('median'))
figuras.fig_ranking_cadenas()

## Hallazgo 4 — Geografía

In [ ]:
p = canast[canast.fecha == canast.fecha.max()].groupby('provincia')['canasta_tipica'].mean().sort_values()
display(p)
print(f"Brecha entre la provincia más cara y la más barata: {100*(p.max()/p.min()-1):.1f}%")
figuras.fig_mapa_provincias()

## Contraste con el IPC del INDEC (opcional, pero es lo que te diferencia)

Bajá el IPC mensual de nivel general y alimentos de
[INDEC](https://www.indec.gob.ar/indec/web/Nivel4-Tema-3-5-31) y compará contra
la variación de tu canasta en el mismo período.

Cuidado con la trampa: si tu canasta se mueve distinto al IPC, la explicación
más probable NO es que el INDEC esté mal, sino que tu canasta pondera distinto
y cubre solo supermercados grandes. Decir eso vos mismo, antes de que te lo
pregunten, es lo que separa a un analista de alguien que corre queries.

In [ ]:
# ipc = pd.read_csv('../data/raw/ipc_indec.csv')
# comparacion = ...


## Las tres frases del pitch

Completá con tus números y practicá decirlas en voz alta:

1. *"El mismo producto, en la misma provincia, cuesta hasta ___% más según la
   cadena. En la mediana de la canasta la brecha es de ___%."*
2. *"Para un hogar tipo eso son $______ por mes, un ___% de la canasta."*
3. *"La cadena más barata está ___ puntos por debajo de la mediana del mercado;
   la más cara, ___ puntos por encima."*

Y la pregunta que te van a hacer: **"¿qué harías con más tiempo?"**
→ Orquestar la ingesta diaria (Airflow/Prefect), tests de calidad automáticos
   (Great Expectations), alertas cuando una cadena deja de reportar, y matcheo
   de productos por embeddings en vez de regex.
